# venture-coder: a LoRA on the engineer's own refusals

Trains `qwen2.5-coder:14b` on repair examples generated by this project's own
gate: the prompt the model sees on a retry -- task, rules, its own refused
answer, the exact selene / luau-lsp / StyLua output that refused it -- answered
with code that passed.

**Before running**, on the game PC:

```bash
python scripts/build_training_set.py
```

and upload `data/training/repairs.jsonl` when the second cell asks.

**Runtime:** Runtime -> Change runtime type -> GPU. A free T4 (16 GB) works but
is tight at this sequence length; L4 or A100 on Colab Pro is comfortable. The
cell below prints which one you got and what that means.

Each example is roughly 6,000 tokens, because the system prompt and the
compiler output are both part of what the model must learn to read. That is why
`MAX_SEQ_LENGTH` is 8192 and not the usual 2048.

In [ ]:
import torch

name = torch.cuda.get_device_name(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {gb:.1f} GB")
if gb < 20:
    print("\nTight. If the training cell runs out of memory, in order:")
    print("  1. MAX_SEQ_LENGTH = 4096 (drops the longest examples, see the filter cell)")
    print("  2. a 7B base instead of 14B")
    print("  3. Colab Pro for an L4 or A100")

In [ ]:
%pip install -q unsloth
%pip install -q --no-deps --upgrade "trl<0.21.0" peft accelerate bitsandbytes

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick data/training/repairs.jsonl
DATA = next(iter(uploaded))
print(DATA)

In [ ]:
import json

from datasets import Dataset

rows = [json.loads(line) for line in open(DATA, encoding="utf-8") if line.strip()]
print(f"{len(rows)} examples")

# Every example must have exactly one assistant turn, and it must be the fix.
# app/engineer/dataset.py guarantees this; check it here anyway, because if it
# ever stops being true `train_on_responses_only` below would quietly train the
# model to reproduce broken code.
for row in rows:
    assistants = [m for m in row["messages"] if m["role"] == "assistant"]
    assert len(assistants) == 1, f"{row['metadata']}: {len(assistants)} assistant turns"

from collections import Counter

print(Counter(row["metadata"]["fix_source"] for row in rows))
print(Counter(row["metadata"]["model"] for row in rows))
dataset = Dataset.from_list([{"messages": row["messages"]} for row in rows])

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 12288
BASE = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
def render(batch):
    return {"text": [tokenizer.apply_chat_template(messages, tokenize=False)
                     for messages in batch["messages"]]}

rendered = dataset.map(render, batched=True, remove_columns=["messages"])

lengths = [len(tokenizer(text).input_ids) for text in rendered["text"]]
lengths.sort()
print(f"tokens: min {lengths[0]}, median {lengths[len(lengths) // 2]}, max {lengths[-1]}")

too_long = sum(1 for n in lengths if n > MAX_SEQ_LENGTH)
if too_long:
    # A truncated example loses its answer and teaches nothing, so drop it
    # rather than train on a cut-off fix.
    print(f"dropping {too_long} example(s) longer than {MAX_SEQ_LENGTH} tokens")
    keep = [len(tokenizer(t).input_ids) <= MAX_SEQ_LENGTH for t in rendered["text"]]
    rendered = rendered.select([i for i, k in enumerate(keep) if k])
print(f"{len(rendered)} examples for training")

In [ ]:
from transformers import DataCollatorForSeq2Seq
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=rendered,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        warmup_steps=5, num_train_epochs=3, learning_rate=1e-4,
        logging_steps=1, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs", report_to="none"))

# Train on the fix only. Without this the model is also trained on the prompt,
# which contains its own refused answer -- the exact code we do not want back.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n")

In [ ]:
# Read one masked example before spending an hour on it: everything except the
# fix should print as blanks. If the broken code is visible here, the mask is
# wrong and the run is worse than useless.
row = trainer.train_dataset[0]
visible = tokenizer.decode([t for t in row["labels"] if t != -100])
print(visible[:2000])

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
# GGUF for Ollama. q4_k_m keeps it about the size the 14B already is on disk.
model.save_pretrained_gguf("venture-coder", tokenizer, quantization_method="q4_k_m")
!ls -lh venture-coder

## Back on the game PC

Download the `.gguf` (it is several GB -- Drive is easier than the browser), put
it next to `training/Modelfile`, then:

```bash
ollama create venture-coder:14b -f training/Modelfile
```

`engineer_ollama_models` already names `venture-coder:14b` first, so the next
run uses it with no code change, and falls back to stock qwen if it is missing.

## How to tell whether it worked

Not by the loss. Run the engineer on the same tasks and compare:

```bash
python -m app.engineer.cli run tasks/BaseHealth.json
```

The numbers that matter are attempts-to-green and the mix of failures
(`data/training/attempts/`). The stock model's baseline is in git history: six
systems, 6/6 attempts refused each, failing on parse errors, invented globals
such as `isfinite`, unused locals and type errors.